# 01 – Data Fetch

This notebook downloads the **Sleep-EDF Expanded (Age subset)** dataset using `mne.datasets.sleep_physionet`.

The objective is to retrieve **Night 2 recordings only** for healthy subjects, as Night 2 sessions are generally more complete and consistent for downstream analysis.

---

## Dataset Scope

- Source: PhysioNet Sleep-EDF Expanded (Age subset)
- Subjects: 0–65 (configurable)
- Recording: Night 2 only

Each successful subject download returns:

- PSG EDF file (signal data)
- Hypnogram EDF file (sleep stage annotations)

---

## Missing Data Handling

Not all subjects have a Night 2 recording available in the corpus.

The notebook:

- Attempts to download Night 2 only
- Skips subjects where Night 2 is unavailable
- Continues execution without interruption
- Tracks missing subjects explicitly

Subjects without Night 2 are reported at the end of execution.

---

## Output

After execution:

- Valid Night 2 PSG + Hypnogram file pairs are available locally
- A summary is printed including:
  - Total valid Night 2 recordings
  - List of subjects missing Night 2

These files are used as input for subsequent preprocessing and relabeling steps.

In [ ]:
import mne
import os
from pathlib import Path
from mne.datasets.sleep_physionet.age import fetch_data

In [ ]:
# --- Configuration for data paths and logging ---

# Project-relative data root
DATA_DIR = (Path.cwd() / ".." / "data" ).resolve()
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data directory {DATA_DIR} does not exist. Please create it before running this notebook.")

# Sleep-EDF directory used by this project (matches downstream notebooks)
SLEEP_EDF_DIR = DATA_DIR / "physionet-sleep-data"
if not SLEEP_EDF_DIR.exists():
    raise FileNotFoundError(f"Sleep-EDF directory {SLEEP_EDF_DIR} does not exist. Please create it before running this notebook.")  

# Report directory for this notebook (matches downstream notebooks)
REPORT_DIR = (Path.cwd() / ".." / "data" / "reports").resolve()
if not REPORT_DIR.exists():
    raise FileNotFoundError(f"Report directory {REPORT_DIR} does not exist. Please create it before running this notebook.")

NOTEBOOK_NAME = "01_data_fetch" # This should match the name of this notebook (without the .ipynb extension)
LOG_FILE = REPORT_DIR / f"{NOTEBOOK_NAME}.txt"

print(f"Data folder: {DATA_DIR}")
print(f"Sleep-EDF folder: {SLEEP_EDF_DIR}")
print(f"Report folder: {REPORT_DIR}")
print(f"Log file: {LOG_FILE}")

In [ ]:
# Configure MNE to store datasets in our project-local raw directory
mne.set_config("MNE_DATA", str(DATA_DIR), set_env=True)
print(f"MNE_DATA set to: {DATA_DIR}")

In [ ]:
# Downloading data for healthy subjects for Night 2 recordings only, which are typically more complete and consistent for analysis.
subject_ids = range(0, 66) # Adjust this range as needed for more subjects
downloaded_files = []
valid_subjects = []
missing = []

for subject in subject_ids:
    try:
        all_files = fetch_data(
            subjects=[subject],
            recording=[2],
            on_missing="warn"
        )

        if not all_files:
            missing.append(subject)
            print("#", end="")
            continue

        downloaded_files.extend(all_files)
        valid_subjects.append(subject)
        print("#", end="")

    except Exception:
        missing.append(subject)
        print("!", end="")

print(f"\nTotal subjects checked: {len(subject_ids)}")
print(f"Total available subjects: {len(valid_subjects)}")
print(f"Rejected '{len(missing)}' subjects with ID: {missing}")

In [ ]:
# Get a list of all files
all_files = os.listdir(SLEEP_EDF_DIR)
psg_files = sorted([f for f in all_files if f.endswith("2E0-PSG.edf")])

# Counter for total subjects
subject_count = 0

with open(LOG_FILE, "w") as f:
    header = f"{'Subject':<10} {'PSG Status':<15} {'Hypnogram Status':<20}"
    separator = "-" * 45

    f.write(header + "\n")
    f.write(separator + "\n")

    for psg in psg_files:
        subject_id = psg[:5]
        hypno_match = [h for h in all_files if h.startswith(subject_id) and "Hypnogram" in h]
        
        psg_status = "ok"
        hypno_status = "ok" if len(hypno_match) > 0 else "MISSING"
        
        row = f"{subject_id:<10} {psg_status:<15} {hypno_status:<20}"
        f.write(row + "\n")
        
        # Increment count
        subject_count += 1

    # Add the total at the bottom
    total_line = f"Total Subjects Found: {subject_count}"
    print(total_line)
    f.write("-" * 45 + "\n")
    f.write(total_line + "\n")

print(f"\nSummary report saved to: {os.path.abspath(LOG_FILE)}")